# LLM Confidence Calibration Pipeline (Scripts 03-05)
**Runtime → Change runtime type → GPU (T4)**

Before running, upload `sft_lora_adapter.zip` to Colab (drag & drop into the file panel on the left).

In [ ]:
# Step 0a: Unzip the SFT adapter
!unzip -o sft_lora_adapter.zip

In [ ]:
# Step 0b: Verify GPU and upload check
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_global_mem / 1e9:.1f} GB")
    device = torch.device("cuda")
    dtype = torch.float16  # GPU can use fp16 for speed
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")
    device = torch.device("cpu")
    dtype = torch.float32

import os
assert os.path.exists("sft_lora_adapter"), "Upload sft_lora_adapter/ folder first! Use the file panel on the left."

In [ ]:
# Step 0b: Verify GPU and upload check
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    device = torch.device("cuda")
    dtype = torch.float16  # GPU can use fp16 for speed
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")
    device = torch.device("cpu")
    dtype = torch.float32

import os
assert os.path.exists("sft_lora_adapter"), "Upload sft_lora_adapter/ folder first! Use the file panel on the left."

---
## Script 03: Create DPO Pairs
Runs SFT model on 2000 NEW TriviaQA questions, uses multi-sample consistency to estimate Brier-optimal confidence.

In [ ]:
import torch
import json
import re
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

model_name = "microsoft/phi-2"
num_samples = 8
temperature = 0.7
num_questions = 2000
question_offset = 2000

# Load tokenizer and SFT model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
print("Loading SFT LoRA adapter...")
model = PeftModel.from_pretrained(base_model, "./sft_lora_adapter")
model = model.to(device)
model.eval()
print("Model loaded on", device)

In [ ]:
def check_answer(model_ans, correct_ans):
    model_ans = model_ans.lower().strip()
    model_ans = model_ans.split("\n")[0].strip()
    for alias in correct_ans:
        alias_clean = alias.lower().strip()
        if alias_clean in model_ans or model_ans.split(".")[0].strip() in alias_clean:
            return 1
    return 0

def extract_answer_text(generated_text):
    text = generated_text.split("\n")[0].strip()
    text = re.sub(r'\s*\[Conf:\d+\.\d+\]', '', text).strip()
    return text

# Load TriviaQA
dataset = load_dataset("trivia_qa", "unfiltered", split="train")
dpo_questions = dataset.select(range(question_offset, question_offset + num_questions))
print(f"Loaded {len(dpo_questions)} questions")

In [ ]:
# Generate DPO pairs
dpo_pairs = []

for i, example in enumerate(dpo_questions):
    question = example["question"]
    aliases = example["answer"]["aliases"]
    prompt = f"Question: {question},\n Answer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    correct_counts = 0
    with torch.no_grad():
        for s in range(num_samples):
            output = model.generate(
                **inputs, max_new_tokens=30,
                do_sample=True, temperature=temperature, top_p=0.95,
            )
            gen_ids = output[0, inputs.input_ids.shape[1]:]
            gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
            correct_counts += check_answer(gen_text, aliases)

    empirical_accuracy = correct_counts / num_samples
    optimal_conf = max(0.05, min(0.95, empirical_accuracy))

    # Greedy decode for canonical answer
    with torch.no_grad():
        greedy_output = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    greedy_ids = greedy_output[0, inputs.input_ids.shape[1]:]
    greedy_text = tokenizer.decode(greedy_ids, skip_special_tokens=True)
    answer_text = extract_answer_text(greedy_text)
    greedy_correct = check_answer(greedy_text, aliases)

    loser_conf = max(0.05, min(0.95, 1.0 - optimal_conf))
    winner = f"{answer_text} [Conf:{optimal_conf:.2f}]"
    loser = f"{answer_text} [Conf:{loser_conf:.2f}]"

    dpo_pairs.append({"prompt": prompt, "winner": winner, "loser": loser})

    if i % 100 == 0:
        print(f"[{i}/{num_questions}] Ans: {answer_text[:30]} | Correct: {greedy_correct} | "
              f"Emp acc: {empirical_accuracy:.2f} | W conf: {optimal_conf:.2f}")

print(f"\nDone! Created {len(dpo_pairs)} DPO pairs.")

In [ ]:
# Save DPO pairs
with open("dpo_pairs.json", "w") as f:
    json.dump(dpo_pairs, f, indent=2)

confs = [float(p["winner"].split("Conf:")[1].rstrip("]")) for p in dpo_pairs]
print(f"Winner confidence stats: mean={np.mean(confs):.3f}, min={np.min(confs):.2f}, "
      f"max={np.max(confs):.2f}, std={np.std(confs):.3f}")

# Free memory before DPO training
del model, base_model
torch.cuda.empty_cache()

---
## Script 04: DPO Training
Train policy model with DPO loss using winner/loser pairs. Reference model stays frozen.

In [ ]:
import torch
import torch.nn.functional as F
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import PeftModel
from torch.utils.data import Dataset, DataLoader

model_name = "microsoft/phi-2"
max_seq_len = 256
batch_size = 4
grad_accum = 4
epochs = 3
lr = 5e-5
warmup_ratio = 0.03
beta = 0.1

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load policy model (trainable)
print("Loading policy model...")
policy_base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
policy_model = PeftModel.from_pretrained(policy_base, "./sft_lora_adapter")
policy_model = policy_model.to(device)
for name, param in policy_model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True

# Load reference model (frozen)
print("Loading reference model...")
ref_base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
ref_model = PeftModel.from_pretrained(ref_base, "./sft_lora_adapter")
ref_model = ref_model.to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

print("Both models loaded on", device)

In [ ]:
# Load DPO pairs
with open("dpo_pairs.json", "r") as f:
    dpo_data = json.load(f)
print(f"Loaded {len(dpo_data)} DPO pairs")

class DPODataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.examples = []
        for item in data:
            prompt = item["prompt"]
            winner = item["winner"]
            loser = item["loser"]
            prompt_enc = tokenizer(prompt, return_tensors="pt")
            prompt_len = prompt_enc.input_ids.shape[1]

            winner_enc = tokenizer(
                prompt + " " + winner, truncation=True,
                max_length=max_len, padding="max_length", return_tensors="pt")
            loser_enc = tokenizer(
                prompt + " " + loser, truncation=True,
                max_length=max_len, padding="max_length", return_tensors="pt")

            self.examples.append({
                "winner_input_ids": winner_enc.input_ids.squeeze(0),
                "winner_attention_mask": winner_enc.attention_mask.squeeze(0),
                "loser_input_ids": loser_enc.input_ids.squeeze(0),
                "loser_attention_mask": loser_enc.attention_mask.squeeze(0),
                "prompt_len": prompt_len,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

dpo_dataset = DPODataset(dpo_data, tokenizer, max_seq_len)
dataloader = DataLoader(dpo_dataset, batch_size=batch_size, shuffle=True)
print("Dataset ready")

In [ ]:
def compute_target_logprobs(model, input_ids, attention_mask, prompt_len):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits.float()  # cast to float32 for stable log_softmax
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    shift_mask = attention_mask[:, 1:].clone()
    for b in range(shift_mask.shape[0]):
        shift_mask[b, :prompt_len - 1] = 0
    log_probs = F.log_softmax(shift_logits, dim=-1)
    gathered = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    target_logprobs = (gathered * shift_mask.float()).sum(dim=-1)
    return target_logprobs

# Optimizer and scheduler
trainable_params = [p for p in policy_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
total_steps = (len(dataloader) // grad_accum) * epochs
warmup_steps = int(total_steps * warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"Total training steps: {total_steps}")

In [ ]:
# DPO Training loop
policy_model.train()
global_step = 0

for epoch in range(epochs):
    epoch_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        w_ids = batch["winner_input_ids"].to(device)
        w_mask = batch["winner_attention_mask"].to(device)
        l_ids = batch["loser_input_ids"].to(device)
        l_mask = batch["loser_attention_mask"].to(device)
        prompt_len = batch["prompt_len"][0].item()

        with torch.no_grad():
            ref_w_logprobs = compute_target_logprobs(ref_model, w_ids, w_mask, prompt_len)
            ref_l_logprobs = compute_target_logprobs(ref_model, l_ids, l_mask, prompt_len)

        policy_w_logprobs = compute_target_logprobs(policy_model, w_ids, w_mask, prompt_len)
        policy_l_logprobs = compute_target_logprobs(policy_model, l_ids, l_mask, prompt_len)

        margin = beta * (
            (policy_w_logprobs - ref_w_logprobs) - (policy_l_logprobs - ref_l_logprobs)
        )
        loss = -F.logsigmoid(margin).mean() / grad_accum
        loss.backward()

        epoch_loss += loss.item() * grad_accum
        num_batches += 1

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            if global_step % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Step {global_step} | Loss: {epoch_loss/num_batches:.4f}")

    if (step + 1) % grad_accum != 0:
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1}/{epochs} complete | Avg Loss: {epoch_loss/num_batches:.4f}")

# Save
policy_model.save_pretrained("./dpo_lora_adapter")
tokenizer.save_pretrained("./dpo_lora_adapter")
print("Saved DPO LoRA adapter to ./dpo_lora_adapter/")

In [ ]:
# DPO Sanity check
policy_model.eval()
test_qs = [
    "What is the capital of France?",
    "Who wrote Romeo and Juliet?",
    "What is the largest planet in our solar system?",
    "Who painted the Mona Lisa?",
    "What year did World War II end?",
]
for q in test_qs:
    prompt = f"Question: {q},\n Answer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = policy_model.generate(**inputs, max_new_tokens=40, do_sample=False)
    gen_text = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {gen_text.strip()}\n")

# Free memory
del policy_model, policy_base, ref_model, ref_base
torch.cuda.empty_cache()

---
## Script 05: Evaluation
Compare Base Phi-2, SFT, and DPO models on 1000 held-out questions.

In [ ]:
import torch
import math
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

model_name = "microsoft/phi-2"
num_eval = 1000
eval_offset = 4000
num_bins = 10

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("trivia_qa", "unfiltered", split="train")
eval_questions = dataset.select(range(eval_offset, eval_offset + num_eval))
print(f"Loaded {num_eval} eval questions")

def check_answer(model_ans, aliases):
    model_ans = model_ans.lower().strip().split("\n")[0].strip()
    for alias in aliases:
        alias_clean = alias.lower().strip()
        if alias_clean in model_ans or model_ans.split(".")[0].strip() in alias_clean:
            return 1
    return 0

def extract_conf_from_text(text):
    match = re.search(r'\[Conf:(\d+\.\d+)\]', text)
    return float(match.group(1)) if match else 0.50

def compute_logprob_confidence(model, tokenizer, prompt, device, max_new_tokens=30):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            output_scores=True, return_dict_in_generate=True, do_sample=False)
    gen_ids = outputs.sequences[0, inputs.input_ids.shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    token_probs = []
    for step, scores in enumerate(outputs.scores):
        probs = torch.softmax(scores.float(), dim=-1)
        chosen = outputs.sequences[0, inputs.input_ids.shape[1] + step]
        token_probs.append(probs[0, chosen].item())
    valid = [p for p in token_probs if p > 0]
    if valid:
        confidence = math.exp(sum(math.log(p) for p in valid) / len(valid))
    else:
        confidence = 0.50
    return gen_text, confidence

def generate_with_conf(model, tokenizer, prompt, device, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_text = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    confidence = extract_conf_from_text(gen_text)
    return gen_text, confidence

def compute_metrics(results):
    confidences = [r["confidence"] for r in results]
    corrects = [r["correct"] for r in results]
    n = len(results)
    bins = np.linspace(0, 1, num_bins + 1)
    ece = 0.0
    bin_data = []
    for b in range(num_bins):
        mask = [(c >= bins[b] and c < bins[b+1]) for c in confidences]
        bc = [c for c, m in zip(corrects, mask) if m]
        bf = [c for c, m in zip(confidences, mask) if m]
        if bc:
            avg_acc = np.mean(bc)
            avg_conf = np.mean(bf)
            ece += (len(bc) / n) * abs(avg_acc - avg_conf)
            bin_data.append({"midpoint": (bins[b]+bins[b+1])/2, "accuracy": avg_acc, "count": len(bc)})
    brier = np.mean([1 - (c - y)**2 for c, y in zip(confidences, corrects)])
    wrong_high = sum(1 for c, y in zip(confidences, corrects) if y == 0 and c > 0.50)
    total_wrong = sum(1 for y in corrects if y == 0)
    overconf = wrong_high / total_wrong if total_wrong > 0 else 0.0
    return {"ece": ece, "brier": brier, "overconfidence_rate": overconf,
            "accuracy": np.mean(corrects), "n": n, "bin_data": bin_data}

def run_eval(model, label, device, use_logprob=False):
    print(f"\n{'='*50}\nEvaluating: {label}\n{'='*50}")
    results = []
    for i, ex in enumerate(eval_questions):
        prompt = f"Question: {ex['question']},\n Answer:"
        aliases = ex["answer"]["aliases"]
        if use_logprob:
            gen_text, conf = compute_logprob_confidence(model, tokenizer, prompt, device)
        else:
            gen_text, conf = generate_with_conf(model, tokenizer, prompt, device)
        correct = check_answer(gen_text, aliases)
        results.append({"question": ex["question"], "generated": gen_text.split("\n")[0].strip(),
                        "confidence": conf, "correct": correct})
        if i % 100 == 0:
            print(f"  [{i}/{num_eval}] Conf: {conf:.3f} | Correct: {correct}")
    metrics = compute_metrics(results)
    print(f"  Accuracy: {metrics['accuracy']:.4f} | ECE: {metrics['ece']:.4f} | "
          f"Brier: {metrics['brier']:.4f} | Overconf: {metrics['overconfidence_rate']:.4f}")
    return results, metrics

def abstention_sweep(results):
    thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
    sweep = []
    for t in thresholds:
        answered = [r for r in results if r["confidence"] >= t]
        coverage = len(answered) / len(results)
        acc = np.mean([r["correct"] for r in answered]) if answered else 0.0
        sweep.append({"threshold": t, "coverage": coverage, "accuracy": acc})
    return sweep

In [ ]:
# Checkpoint 1: Base Phi-2
print("Loading base Phi-2...")
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device)
base_results, base_metrics = run_eval(base_model, "Base Phi-2", device, use_logprob=True)
base_sweep = abstention_sweep(base_results)
del base_model
torch.cuda.empty_cache()

In [ ]:
# Checkpoint 2: SFT model
print("Loading SFT model...")
sft_base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
sft_model = PeftModel.from_pretrained(sft_base, "./sft_lora_adapter").to(device)
sft_model.eval()
sft_results, sft_metrics = run_eval(sft_model, "SFT Model", device)
sft_sweep = abstention_sweep(sft_results)
del sft_model, sft_base
torch.cuda.empty_cache()

In [ ]:
# Checkpoint 3: DPO model
print("Loading DPO model...")
dpo_base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
dpo_model = PeftModel.from_pretrained(dpo_base, "./dpo_lora_adapter").to(device)
dpo_model.eval()
dpo_results, dpo_metrics = run_eval(dpo_model, "DPO Model", device)
dpo_sweep = abstention_sweep(dpo_results)
del dpo_model, dpo_base
torch.cuda.empty_cache()

In [ ]:
# Save results
import os
os.makedirs("results", exist_ok=True)

eval_output = {
    "base": {"metrics": {k:v for k,v in base_metrics.items() if k!="bin_data"},
             "bin_data": base_metrics["bin_data"], "sweep": base_sweep},
    "sft":  {"metrics": {k:v for k,v in sft_metrics.items() if k!="bin_data"},
             "bin_data": sft_metrics["bin_data"], "sweep": sft_sweep},
    "dpo":  {"metrics": {k:v for k,v in dpo_metrics.items() if k!="bin_data"},
             "bin_data": dpo_metrics["bin_data"], "sweep": dpo_sweep},
}
with open("results/evaluation_results.json", "w") as f:
    json.dump(eval_output, f, indent=2)

# Calibration plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot([0,1], [0,1], 'k--', label='Perfect')
for label, metrics, color in [("Base", base_metrics, "red"), ("SFT", sft_metrics, "blue"), ("DPO", dpo_metrics, "green")]:
    if metrics["bin_data"]:
        mids = [b["midpoint"] for b in metrics["bin_data"]]
        accs = [b["accuracy"] for b in metrics["bin_data"]]
        ax.plot(mids, accs, 'o-', color=color, label=f'{label} (ECE={metrics["ece"]:.3f})', markersize=8)
ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.set_title('Calibration Curves')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for label, sweep, color in [("Base", base_sweep, "red"), ("SFT", sft_sweep, "blue"), ("DPO", dpo_sweep, "green")]:
    ax.plot([s["coverage"] for s in sweep], [s["accuracy"] for s in sweep], 'o-', color=color, label=label)
ax.set_xlabel('Coverage'); ax.set_ylabel('Accuracy'); ax.set_title('Abstention Sweep')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
labels = ['Base', 'SFT', 'DPO']
x = np.arange(3); w = 0.35
ax.bar(x-w/2, [base_metrics["ece"], sft_metrics["ece"], dpo_metrics["ece"]], w, label='ECE', color='steelblue')
ax.bar(x+w/2, [base_metrics["overconfidence_rate"], sft_metrics["overconfidence_rate"], dpo_metrics["overconfidence_rate"]], w, label='Overconf', color='salmon')
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_title('ECE & Overconfidence')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("results/calibration_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# Final summary
print("\n" + "="*60 + "\nFINAL SUMMARY\n" + "="*60)
print(f"{'Metric':<25} {'Base':>10} {'SFT':>10} {'DPO':>10}")
print("-"*55)
for m in ['accuracy', 'ece', 'brier', 'overconfidence_rate']:
    print(f"{m:<25} {base_metrics[m]:>10.4f} {sft_metrics[m]:>10.4f} {dpo_metrics[m]:>10.4f}")